# Importing and preprocessing the data
The data is first imported from the SQLite file and converted into a pandas dataframe. After that, all N/A and duplicate records are dropped, and a the dataset is subsetted randomly. 

## Importing the data

In [1]:
import sqlite3
import pandas as pd
import numpy as np

conn = sqlite3.connect("./data/properties_cleaned.db")
df = pd.read_sql_query(f"SELECT * FROM properties", conn)
conn.close()

#['bedrooms', 'building_form', 'city', 'commercial_space',
# 'days_on_market', 'energy_efficient', 'energy_label', 'fixer_upper',
# 'has_balcony', 'has_garden', 'has_heat_pump', 'has_roof_terrace',
# 'has_solar_panels', 'latitude', 'living_area', 'longitude',
# 'national_monument', 'object_type', 'plot_area_m2', 'postcode4',
# 'price', 'province', 'publication_date', 'rooms', 'url', 'year_built',
# 'has_basement', 'has_attic', 'stories', 'addresses_per_km2', 'urbanization_level'],

df = df.dropna()
df = df.drop_duplicates()
#df = df.sample(n=10000, random_state=1)

# Select subset
df = df[['price', 'bedrooms', 'building_form', 
         #'city', 
         'commercial_space',
         'days_on_market', 
         'energy_efficient', 'energy_label', 'fixer_upper',
         'has_balcony', 'has_garden', 'has_heat_pump', 'has_roof_terrace',
         'has_solar_panels', 'latitude', 'living_area', 'longitude',
         'national_monument', 
         'object_type', 'plot_area_m2', #'postcode4',
         'province', 
         'publication_date', 
         #'urbanization_level',
         #'publication_date',
         'addresses_per_km2',
         'rooms', 'year_built',
         'has_basement', 'has_attic', 'stories']]

## Converting datatypes
All columns have to be converted to the proper datatypes to allow modeling

In [2]:
# CATEGORY
df['building_form'] = df['building_form'].astype('category')
df['object_type'] = df['object_type'].astype('category')
df['province'] = df['province'].astype('category')

# Ordinal category for energy_label
df['energy_label'] = pd.Categorical(
    df['energy_label'],
    categories=['g', 'f', 'e', 'd', 'c', 'b', 'a'],
    ordered=True
)

# BOOL
df['has_balcony'] = df['has_balcony'].astype(bool)
df['has_garden'] = df['has_garden'].astype(bool)
df['has_heat_pump'] = df['has_heat_pump'].astype(bool)
df['has_roof_terrace'] = df['has_roof_terrace'].astype(bool)
df['has_solar_panels'] = df['has_solar_panels'].astype(bool)
df['has_basement'] = df['has_basement'].astype(bool)
df['has_attic'] = df['has_attic'].astype(bool)
df['fixer_upper'] = df['fixer_upper'].astype(bool)
df['energy_efficient'] = df['energy_efficient'].astype(bool)
df['commercial_space'] = df['commercial_space'].astype(bool)
df['national_monument'] = df['national_monument'].astype(bool)

# INT
df['rooms'] = df['rooms'].astype(int)
df['year_built'] = df['year_built'].astype(int)
df['living_area'] = df['living_area'].astype(int)
df['plot_area_m2'] = df['plot_area_m2'].astype(int)
df['days_on_market'] = df['days_on_market'].astype(int)
df['bedrooms'] = df['bedrooms'].astype(int)
df['stories'] = df['stories'].astype(int)
df['addresses_per_km2'] = df['addresses_per_km2'].astype(int)
#df['urbanization_level'] = df['urbanization_level'].astype(int)

# FLOAT
df['latitude'] = df['latitude'].astype(float)
df['longitude'] = df['longitude'].astype(float)
df['price'] = df['price'].str.replace('€', '').str.replace('.', '').astype(float)

# DATE
# Remove duplicate 'publication_date' column if it exists
if df.columns.duplicated().any():
    df = df.loc[:, ~df.columns.duplicated()]

# Convert to datetime
df['publication_date'] = pd.to_datetime(df['publication_date'], format='%Y-%m-%d')

# Extract useful numeric features from publication_date
df['pub_year'] = df['publication_date'].dt.year
df['pub_month'] = df['publication_date'].dt.month
df['pub_dayofweek'] = df['publication_date'].dt.dayofweek
df['pub_dayofyear'] = df['publication_date'].dt.dayofyear

# Drop the original 'publication_date' column
df = df.drop(columns=['publication_date'])

# print data types
print(df.dtypes)

price                 float64
bedrooms                int64
building_form        category
commercial_space         bool
days_on_market          int64
energy_efficient         bool
energy_label         category
fixer_upper              bool
has_balcony              bool
has_garden               bool
has_heat_pump            bool
has_roof_terrace         bool
has_solar_panels         bool
latitude              float64
living_area             int64
longitude             float64
national_monument        bool
object_type          category
plot_area_m2            int64
province             category
addresses_per_km2       int64
rooms                   int64
year_built              int64
has_basement             bool
has_attic                bool
stories                 int64
pub_year                int32
pub_month               int32
pub_dayofweek           int32
pub_dayofyear           int32
dtype: object


In [3]:
print(len(df))

def remove_outliers_iqr(data, column):
    Q1 = data[column].quantile(0.25)
    Q3 = data[column].quantile(0.75)
    IQR = Q3 - Q1
    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR
    return data[(data[column] >= lower_bound) & (data[column] <= upper_bound)]

# Remove outliers for specified columns
for col in ['bedrooms', 'living_area', 'plot_area_m2', 'rooms', 'year_built', 'stories', 'price']:
    df = remove_outliers_iqr(df, col)
print(len(df))

144662
120601


## Check for Multicolinearity
Multicollinearity doesnt impact rf, but important for interpretting 

In [4]:
import pandas as pd
from statsmodels.stats.outliers_influence import variance_inflation_factor
from statsmodels.tools.tools import add_constant

numeric_vars = df.select_dtypes(include=['number']).drop(columns='price')
X = add_constant(numeric_vars)  # A constant is added
 
# Compute VIF for each numeric feature
vif_data = pd.DataFrame()
vif_data['feature'] = X.columns
vif_data['VIF'] = [variance_inflation_factor(X.values, i) for i in range(X.shape[1])]
vif_data = vif_data.sort_values(by='VIF', ascending=False)  # Sort by VIF values

print(vif_data)

/Library/Frameworks/Python.framework/Versions/3.9/lib/python3.9/site-packages/statsmodels/stats/outliers_influence.py:197: RuntimeWarning: divide by zero encountered in scalar divide
  vif = 1. / (1. - r_squared_i)
/Library/Frameworks/Python.framework/Versions/3.9/lib/python3.9/site-packages/statsmodels/regression/linear_model.py:1782: RuntimeWarning: divide by zero encountered in scalar divide
  return 1 - self.ssr/self.centered_tss


              feature        VIF
1      days_on_market        inf
13      pub_dayofyear        inf
11          pub_month  89.936147
7               rooms   5.631579
0            bedrooms   5.544435
3         living_area   2.870846
5        plot_area_m2   2.053006
6   addresses_per_km2   1.650655
9             stories   1.587083
8          year_built   1.293995
4           longitude   1.280941
2            latitude   1.130389
12      pub_dayofweek   1.004833
10           pub_year   0.000000


The VIF values show signs of multicolliniearity for bedrooms and rooms. Therefore, the rooms variable will be converted to a non_bedroom_rooms variable. This will give the variable a different meaning, while still maintaining context.

In [5]:
if 'rooms' in df.columns:
    df['non_bedroom_rooms'] = df['rooms'] - df['bedrooms'].astype(int)
    df = df.drop(columns=['rooms'])

# Model creation and performance
Six models are created and evaluated on performance.
- Linear Regression
- KNN
- Random Forest
- XGBoost
- LightGBM

## K-fold cross validation setup

In [6]:
from sklearn.model_selection import KFold
kf = KFold(n_splits=5, shuffle=True, random_state=42)

## Linear Regression (OLS)

In [7]:
import patsy
import statsmodels.api as sm
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import numpy as np

mae_scores, rmse_scores, r2_scores = [], [], []

for train_idx, test_idx in kf.split(df):
    train, test = df.iloc[train_idx], df.iloc[test_idx]
    y_train, y_test = train['price'], test['price']
    X_train = train.drop(columns=['price', 'publication_date']) if 'publication_date' in train.columns else train.drop(columns=['price'])
    X_test = test.drop(columns=['price', 'publication_date']) if 'publication_date' in test.columns else test.drop(columns=['price'])
    
    # Build formula for patsy
    formula = 'price ~ ' + ' + '.join(X_train.columns)
    y_train_sm, X_train_sm = patsy.dmatrices(formula, data=train, return_type='dataframe')
    y_test_sm, X_test_sm = patsy.dmatrices(formula, data=test, return_type='dataframe')
    
    model_cv = sm.OLS(y_train_sm, X_train_sm).fit()
    y_pred = model_cv.predict(X_test_sm)
    
    mae_scores.append(mean_absolute_error(y_test, y_pred))
    rmse_scores.append(np.sqrt(mean_squared_error(y_test, y_pred)))
    r2_scores.append(r2_score(y_test, y_pred))

# Store all scores in a dictionary
lr_scores = {
    'mae': mae_scores,
    'rmse': rmse_scores,
    'r2': r2_scores
}

print(f"MAE: {np.mean(mae_scores):.2f} ± {np.std(mae_scores):.2f}")
print(f"RMSE: {np.mean(rmse_scores):.2f} ± {np.std(rmse_scores):.2f}")
print(f"R^2: {np.mean(r2_scores):.4f} ± {np.std(r2_scores):.4f}")


MAE: 60947.65 ± 97.71
RMSE: 79175.91 ± 92.01
R^2: 0.6547 ± 0.0015


## KNN

In [9]:
from sklearn.neighbors import KNeighborsRegressor
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import numpy as np

y = df['price']
X = pd.get_dummies(df.drop(columns=['price', 'latitude', 'longitude']), drop_first=True)

mae_scores = []
rmse_scores = []
r2_scores = []

for train_idx, test_idx in kf.split(X):
    X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
    y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]
    
    scaler = StandardScaler()
    X_train_scaled = pd.DataFrame(scaler.fit_transform(X_train), columns=X.columns, index=X_train.index)
    X_test_scaled = pd.DataFrame(scaler.transform(X_test), columns=X.columns, index=X_test.index)
    
    knn = KNeighborsRegressor(n_neighbors=11, weights='distance', metric='manhattan')
    knn.fit(X_train, y_train)
    y_pred = knn.predict(X_test)
    
    mae_scores.append(mean_absolute_error(y_test, y_pred))
    rmse_scores.append(np.sqrt(mean_squared_error(y_test, y_pred)))
    r2_scores.append(r2_score(y_test, y_pred))

# Store all scores in a dictionary
knn_scores = {
    'mae': mae_scores,
    'rmse': rmse_scores,
    'r2': r2_scores
}

print(f"MAE: {np.mean(mae_scores):.2f} ± {np.std(mae_scores):.2f}")
print(f"RMSE: {np.mean(rmse_scores):.2f} ± {np.std(rmse_scores):.2f}")
print(f"R^2: {np.mean(r2_scores):.4f} ± {np.std(r2_scores):.4f}")


MAE: 66259.25 ± 201.20
RMSE: 88372.14 ± 354.50
R^2: 0.5698 ± 0.0019


## Random Forest

In [ ]:
from sklearn.model_selection import KFold
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import numpy as np

y = df['price']
X = pd.get_dummies(df.drop(columns=['price']), drop_first=True)

mae_scores = []
rmse_scores = []
r2_scores = []

for train_idx, test_idx in kf.split(X):
    X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
    y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]
    
    rf = RandomForestRegressor(
        n_estimators=100, #100
        #max_depth=25,
        #max_features='sqrt',
        #min_samples_split=5,
        #min_samples_leaf=4,
        random_state=42,
        #n_jobs=-1
    )
    rf.fit(X_train, y_train)
    y_pred = rf.predict(X_test)
    
    mae_scores.append(mean_absolute_error(y_test, y_pred))
    rmse_scores.append(np.sqrt(mean_squared_error(y_test, y_pred)))
    r2_scores.append(r2_score(y_test, y_pred))

# Store all scores in a dictionary
rf_scores = {
    'mae': mae_scores,
    'rmse': rmse_scores,
    'r2': r2_scores
}

print("MAE: {:.2f} ± {:.2f}".format(np.mean(mae_scores), np.std(mae_scores)))
print("RMSE: {:.2f} ± {:.2f}".format(np.mean(rmse_scores), np.std(rmse_scores)))
print("R^2: {:.3f} ± {:.3f}".format(np.mean(r2_scores), np.std(r2_scores)))

## XGBoost

In [34]:
from xgboost import XGBRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import numpy as np
from sklearn.model_selection import KFold
from sklearn.preprocessing import StandardScaler

y = df['price']
X = pd.get_dummies(df.drop(columns=['price']), drop_first=True)

mae_scores = []
rmse_scores = []
r2_scores = []

for train_idx, test_idx in kf.split(X):
    X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
    y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]
    
    xgb = XGBRegressor(
        n_estimators=444,
        max_depth=9,
        learning_rate=0.052277267492428794,
        subsample=0.9947547746402069,
        colsample_bytree=0.7123738038749523,
        gamma=0.16280882494747453,
        min_child_weight=1,
        random_state=42,
        n_jobs=-1
    )
    xgb.fit(X_train, y_train)
    y_pred = xgb.predict(X_test)
    
    mae_scores.append(mean_absolute_error(y_test, y_pred))
    rmse_scores.append(np.sqrt(mean_squared_error(y_test, y_pred)))
    r2_scores.append(r2_score(y_test, y_pred))


# Store all scores in a dictionary
xgb_scores = {
    'mae': mae_scores,
    'rmse': rmse_scores,
    'r2': r2_scores
}

print("MAE (mean ± std): {:.2f} ± {:.2f}".format(np.mean(mae_scores), np.std(mae_scores)))
print("RMSE (mean ± std): {:.2f} ± {:.2f}".format(np.mean(rmse_scores), np.std(rmse_scores)))
print("R^2 (mean ± std): {:.3f} ± {:.3f}".format(np.mean(r2_scores), np.std(r2_scores)))


MAE (mean ± std): 32846.00 ± 79.49
RMSE (mean ± std): 45592.37 ± 152.05
R^2 (mean ± std): 0.885 ± 0.001
MAPE (mean ± std): 0.08 ± 0.08


## LightGBM

In [ ]:
from lightgbm import LGBMRegressor

mae_scores = []
rmse_scores = []
r2_scores = []

for train_idx, test_idx in kf.split(X):
    X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
    y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]

    model = LGBMRegressor(
        bagging_fraction=0.8,
        bagging_freq=0,
        feature_fraction=0.8,
        learning_rate=0.06697167353375243,
        max_depth=15,
        min_data_in_leaf=20,
        min_gain_to_split=0.01,
        n_estimators=508,
        num_leaves=64,
        subsample=1.0,
        random_state=42,
        n_jobs=-1
    )
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)

    mae_scores.append(mean_absolute_error(y_test, y_pred))
    rmse_scores.append(np.sqrt(mean_squared_error(y_test, y_pred)))
    r2_scores.append(r2_score(y_test, y_pred))

# Store all scores in a dictionary
lgbm_scores = {
    'mae': mae_scores,
    'rmse': rmse_scores,
    'r2': r2_scores
}

print("CV MAE: {:.2f} ± {:.2f}".format(np.mean(mae_scores), np.std(mae_scores)))
print("CV RMSE: {:.2f} ± {:.2f}".format(np.mean(rmse_scores), np.std(rmse_scores)))
print("CV R^2: {:.3f} ± {:.3f}".format(np.mean(r2_scores), np.std(r2_scores)))


## Feed Forward Neural Network (MLP)

In [ ]:
from sklearn.model_selection import KFold
from sklearn.neural_network import MLPRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

y = df['price']
X = pd.get_dummies(df.drop(columns=['price']), drop_first=True)

mae_scores = []
rmse_scores = []
r2_scores = []

for train_idx, val_idx in kf.split(X):
    # Scale within the fold
    scaler = StandardScaler()
    X_tr = scaler.fit_transform(X.iloc[train_idx])
    X_val = scaler.transform(X.iloc[val_idx])
    y_tr = y.iloc[train_idx]
    y_val = y.iloc[val_idx]

    nn_cv = MLPRegressor(
        hidden_layer_sizes=(128, 64, 32),
        activation='relu',
        alpha=0.007975429868602328,
        batch_size=78,
        learning_rate_init=0.007896910002727693,
        max_iter=500,
        random_state=42,
        early_stopping=True
    )
    nn_cv.fit(X_tr, y_tr)
    y_val_pred = nn_cv.predict(X_val)

    mae_scores.append(mean_absolute_error(y_val, y_val_pred))
    rmse_scores.append(np.sqrt(mean_squared_error(y_val, y_val_pred)))
    r2_scores.append(r2_score(y_val, y_val_pred))

nn_scores = {
    'mae': mae_scores,
    'rmse': rmse_scores,
    'r2': r2_scores
}

print(f"5-Fold CV MAE: {np.mean(mae_scores):.2f} ± {np.std(mae_scores):.2f}")
print(f"5-Fold CV RMSE: {np.mean(rmse_scores):.2f} ± {np.std(rmse_scores):.2f}")
print(f"5-Fold CV R2: {np.mean(r2_scores):.3f} ± {np.std(r2_scores):.3f}")


## Combine performance metrics and save

In [ ]:
scores = {
    'lr': lr_scores,
    'knn': knn_scores,
    'rf': rf_scores,
    'xgb': xgb_scores,
    'lgbm': lgbm_scores,
    'nn': nn_scores
}

# Save scores to file as json
import json

with open('performance_scores.json', 'w') as f:
    json.dump(scores, f)

# Feature Importance

## SHAP Analysis